<a href="https://colab.research.google.com/github/matthewhawksby/colabnotebooks/blob/main/Copy_of_MNE_DataAnalysis_pt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mne
import mne
import os
import sys
import pickle
import numpy as np
import pandas as pd
import glob

import helper

sys.path.append('/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src')

import helper

#Check if we are in colab.
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

#Load the correct base path
if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/Colab Notebooks/MNE/mne_data'
else:
    base_path = os.path.join(os.getcwd(), 'mne-data', 'MNE-sample-data')


#Open pickle file and load it into 'blocking'
with open('/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/blocking.pkl', 'rb') as f:
    blocking = pickle.load(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

### GET all STI channels
sti_channels = [ch for ch in raw.info['ch_names'] if ch.startswith('STI') or ch.startswith('STI') and ch != 'STI101']
events = mne.find_events(raw, shortest_event=1)
samplingFrequency = raw.info['sfreq']

### Print the detected events
events = [(sample/samplingFrequency, eventValue) for sample, _, eventValue in events]

all_stimuli = []
count = 0
for time, value in events:
    if value >= 512:
        value -= 512
    if value >= 256:
        value -= 256
    if value == 0 or value == 60:
        continue
    if count == 0:
        all_stimuli.append([])
    all_stimuli[-1].append((time, val))
    count = (count + 1) % 3


NameError: name 'raw' is not defined

In [ ]:
### LOAD CSV INTO DATAFRAME
dir = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/data/"
files = glob.glob(dir + "M002_main_2025-04-14_13h48.34.684.csv")
csvDataframe = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Loaded {len(csvDataframe)} trials")


In [ ]:

##Get all tsvs
tsv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
all_data = helper.stimuli_load(tsv_path)

response = {}

for index, row in csvDataframe.iterrows():
    stimuli = str(row["audio_file"]).replace("\\", "/")
    resp = row[response_column]

    if stimuli.lower() == "nan":
        continue

    if str(resp).lower() == "nan":
        resp = -1
    else:
        resp = int(resp)
        if not use_button:
            resp = resp - 1

    response[stimuli] = {}
    response[stimuli]["response"] = resp

for stimuli in response:
    for key in ["speaker", "sibilants", "normality", "perplexity", "tgt_word", "tgt_sound"]:
        if stimuli not in all_data:
          print("Missing:", stimuli)
          print("Closest matches:", [k for k in all_data if row['audio_file'] in k])
          continue  # Skip this row
        response[stimuli][key] = all_data[stimuli][key]


    if "bot" in response[stimuli]["speaker"]:
        response[stimuli]["speaker"] = "bot"
    else:
        response[stimuli]["speaker"] = "human"

    if response[stimuli]["sibilants"] == "nonsibilant":
        sound_pair = ["f", "θ"]
    else:
        sound_pair = ["s", "ʃ"]

    resp = response[stimuli]["response"]
    if resp == -1:
        response[stimuli]["resp_correct"] = None
    elif response[stimuli]["tgt_sound"] == sound_pair[resp]:
        response[stimuli]["resp_correct"] = True
    else:
        response[stimuli]["resp_correct"] = False

In [ ]:
from collections import Counter

### Collect unmatched stimuli
unmatched = []

for stim in response:
    if stim not in all_data:
        unmatched.append(stim)

### Show summary and a preview
print(f"\n Unmatched stimuli: {len(unmatched)}")
if unmatched:
    print("Here are a few examples:")
    for stim in unmatched[:10]:
        print(" -", stim)


### Filter for valid annotated data
cleaned_response = {}

for stim, data in response.items():
    if stim not in all_data:
        continue  # skip if not in master stimuli list
    if "normality" not in data or "resp_correct" not in data:
        continue  # skip incomplete data
    cleaned_response[stim] = data

r = cleaned_response

###
print(f"Total entries in response: {len(response)}")
print(f"Valid/filtered entries: {len(r)}\n")
# -------------------------------------
# Filtering logic
# -------------------------------------
def eval(i, cond):
    if "nor" in cond and i["normality"] != "normal":
        return False
    if "abn" in cond and i["normality"] != "abnormal":
        return False
    if "sib" in cond and i["sibilants"] != "sibilant":
        return False
    if "nos" in cond and i["sibilants"] != "nonsibilant":
        return False
    if "man" in cond and i["speaker"] != "human":
        return False
    if "bot" in cond and i["speaker"] != "bot":
        return False
    if "cor" in cond and i["resp_correct"] is not True:
        return False
    if "err" in cond and i["resp_correct"] is not False:
        return False
    return True

def count(r, cond=[]):
    return sum(1 for v in r.values() if eval(v, cond))

def rate(r, cond=[]):
    num = count(r, ["cor"] + cond)
    denom = count(r, cond)
    if denom == 0:
        print(f"No trials matched condition: {cond}")
        return 0.0
    return num / denom * 100

# -------------------------------------
# Breakdown
# -------------------------------------
print("Normality distribution:")
print(Counter([v["normality"] for v in r.values()]))
print()

print("Total valid trials:", len(r))
print(f"Overall recall: {rate(r, []):.2f}%")

def show_rate(label, cond):
    print(f"{label:<30} {rate(r, cond):>6.2f}%")

print("\n Condition breakdown:")
show_rate("Normal trials", ["nor"])
show_rate("Abnormal trials", ["abn"])
show_rate("Normal + Bot", ["nor", "bot"])
show_rate("Abnormal + Bot", ["abn", "bot"])
show_rate("Normal + Human", ["nor", "man"])
show_rate("Abnormal + Human", ["abn", "man"])

show_rate("Normal + Bot + Sibilant", ["nor", "bot", "sib"])
show_rate("Abnormal + Bot + Sibilant", ["abn", "bot", "sib"])
show_rate("Normal + Human + Sibilant", ["nor", "man", "sib"])
show_rate("Abnormal + Human + Sibilant", ["abn", "man", "sib"])

show_rate("Normal + Bot + Nonsibilant", ["nor", "bot", "nos"])
show_rate("Abnormal + Bot + Nonsibilant", ["abn", "bot", "nos"])
show_rate("Normal + Human + Nonsibilant", ["nor", "man", "nos"])
show_rate("Abnormal + Human + Nonsibilant", ["abn", "man", "nos"])



In [ ]:
# Flatten the blocking.pkl file
stimulus_order = []
for block in blocking:
    for entry in block:
        if isinstance(entry, list):
            stimulus_order.extend([e for e in entry if isinstance(e, str)])
        elif isinstance(entry, str):
            stimulus_order.append(entry)

print("Flattened stimulus count:", len(stimulus_order))
print("Sample from stimulus_order:")
for stim in stimulus_order[:10]:
    print(f"{stim}")

# Load all_data from TSVs
tsv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
all_data = helper.stimuli_load(tsv_path)
print("Entries in all_data:", len(all_data))

#Any missing entries?
missing = []

print(f"\n Unmatched entries: {len(missing)}")
if missing:
    print("Sample of unmatched entries:")
    for m in missing[:5]:
        print(m)



In [ ]:


csv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/data/M002_main_2025-04-14_13h48.34.684.csv"
df = pd.read_csv(csv_path)

df["audio_file"] = df["audio_file"].str.replace("\\", "/", regex=False)
df["audio_file"] = df["audio_file"].str.replace("../", "../", regex=False)

tsv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
all_data = helper.stimuli_load(tsv_path)

meta_df = pd.DataFrame.from_dict(all_data, orient='index').reset_index()
meta_df.rename(columns={"index": "stimulus"}, inplace=True)
df = df.merge(meta_df, how="left", left_on="audio_file", right_on="stimulus")

df["global_onset"] = df["presentation.started"] + df["trigger_word_on.started"]

df["response_clean"] = pd.to_numeric(df["buttonBox_2.keys"], errors='coerce').fillna(-1).astype(int)

df["speaker_type"] = df["speaker"].apply(
    lambda s: "bot" if s in {"bot", "bot2"} else ("human" if pd.notna(s) else None)
)

def compute_resp_correct(row):
    if row["response_clean"] == -1 or pd.isna(row["tgt_sound"]) or pd.isna(row["sibilants"]):
        return None
    options = ["f", "θ"] if row["sibilants"] == "nonsibilant" else ["s", "ʃ"]
    if 0 <= row["response_clean"] < len(options):
        return row["tgt_sound"] == options[row["response_clean"]]
    return None

df["resp_correct"] = df.apply(compute_resp_correct, axis=1)

valid_df = df[df["response_clean"] != -1].copy()
valid_df = valid_df.sort_values("global_onset").reset_index(drop=True)

print(f"Trials in valid DataFrame: {len(valid_df)}")

alignment_df = valid_df
display(alignment_df)


In [ ]:
#Reaction Type by Normal/Abnormal
sns.boxplot(data=alignment_df, x="speaker_type", y="trials.buttonBox_2.rt")
plt.title("Reaction Time by Normality")
plt.show()

In [ ]:
raw_list = []
base_path2 = "/content/drive/MyDrive/Colab Notebooks/MNE/MNE_data/m_002/250414"
for i in range(1):
    fpath = os.path.join(base_path2, f'M002_block{i}_raw.fif')
    raw = mne.io.read_raw_fif(fpath, preload=True, verbose='ERROR')
    raw_list.append(raw)

raw_combined = mne.concatenate_raws(raw_list, on_mismatch='ignore')

In [ ]:
# PARAMETERS
sfreq = raw_combined.info['sfreq']
fixed_window_sec = 0.5  # seconds
fixed_window_samples = int(fixed_window_sec * sfreq)

# Pick MEG channels
meg_picks = mne.pick_types(raw_combined.info, meg=True, stim=False, eog=False, exclude='bads')

# Extract trials
X_raw = []
y = []

for _, row in alignment_df.iterrows():
    if row.get("normality") not in ["normal", "abnormal"]:
        continue
    if pd.isna(row.get("global_onset")):
        continue

    onset_sample = int(row["global_onset"] * sfreq)

    if onset_sample < 0 or onset_sample + fixed_window_samples > raw_combined.n_times:
        continue

    data = raw_combined.get_data(
        picks=meg_picks,
        start=onset_sample,
        stop=onset_sample + fixed_window_samples
    )

    X_raw.append(data[np.newaxis, ...])
    y.append(1 if row["normality"] == "abnormal" else 0)

X_raw = np.concatenate(X_raw, axis=0)
y = np.array(y)

print(f"✅ Extracted {X_raw.shape[0]} trials with shape {X_raw.shape}")


In [ ]:
X_avg = X_raw.mean(axis=-1)

X_normal = X_avg[y == 0]
X_abnormal = X_avg[y == 1]

t_vals = []
p_vals = []

for ch in range(X_avg.shape[1]):
    t_stat, p_val = stats.ttest_ind(X_normal[:, ch], X_abnormal[:, ch], equal_var=False)
    t_vals.append(t_stat)
    p_vals.append(p_val)

t_vals = np.array(t_vals)
p_vals = np.array(p_vals)

plt.figure(figsize=(14, 6))
plt.subplot(1,2,1)
plt.bar(np.arange(len(t_vals)), t_vals)
plt.title("T-statistics per Channel")
plt.xlabel("Channel")
plt.ylabel("T-value")

plt.subplot(1,2,2)
plt.bar(np.arange(len(p_vals)), -np.log10(p_vals))  # Log-scaled p-values
plt.title("-log10(p-value) per Channel")
plt.xlabel("Channel")
plt.ylabel("-log10(p-value)")
plt.tight_layout()
plt.show()

# 5. Print Top Channels
top_ch_idx = np.argsort(p_vals)[:10]
print("🔝 Top channels with smallest p-values:")
for idx in top_ch_idx:
    print(f"Channel {idx} ({raw_combined.ch_names[idx]}): p={p_vals[idx]:.2e}, t={t_vals[idx]:.2f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# =========================================
# 1. Create the combined trials
# =========================================
n_trials, n_channels, n_times = X_raw.shape
half_point = n_times // 2

X_new = []
y_new = []

for i in range(n_trials - 1):
    second_half = X_raw[i, :, half_point:]
    first_half = X_raw[i+1, :, :half_point]
    combined = np.concatenate([second_half, first_half], axis=1)
    X_new.append(combined)
    y_new.append(y[i])  # Label from trial i

X_new = np.stack(X_new)
y_new = np.array(y_new)

print(f"✅ New X shape: {X_new.shape}")
print(f"✅ New y shape: {y_new.shape}")

# =========================================
# 2. Flatten features
# =========================================
X_feat = X_new.reshape(X_new.shape[0], -1)
print(f"✅ Flattened X_feat shape: {X_feat.shape}")

# =========================================
# 3. Scale
# =========================================
scaler = StandardScaler()
X_feat = scaler.fit_transform(X_feat)

# =========================================
# 4. Define Classifiers
# =========================================
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100),
    "SVM (Linear Kernel)": SVC(kernel='linear', probability=True),
    "SVM (RBF Kernel)": SVC(kernel='rbf', probability=True),
    "XGBoost (GPU)": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        device='cuda',  # GPU!
        tree_method='hist',
        eval_metric='logloss'
    )
}

# =========================================
# 5. Train and Evaluate
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X_feat, y_new, stratify=y_new, test_size=0.2, random_state=42
)

for name, clf in classifiers.items():
    print(f"\n🧠 {name}")
    pipe = make_pipeline(clf)

    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    print(f"  ➡️ CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=["Normal", "Abnormal"]))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Abnormal"])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(cmap="Blues", ax=ax, values_format='d')
    plt.title(f"{name} Confusion Matrix")
    plt.grid(False)
    plt.show()


In [ ]:
# ===========================================
# 1. Imports
# ===========================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ===========================================
# 2. Prepare Data
# ===========================================
# Let's use only the first 20 MEG channels to keep it small
# (you can adjust later)
top_n_channels = 50
X_cnn = X_raw[:, :top_n_channels, :]  # (n_trials, top_n_channels, timepoints)

print(f"✅ CNN input shape: {X_cnn.shape}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_cnn, y, test_size=0.2, stratify=y, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create datasets and loaders
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

# ===========================================
# 3. Define CNN Model
# ===========================================
class MEG_CNN(nn.Module):
    def __init__(self, n_channels, n_times):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=n_channels, out_channels=32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool = nn.AdaptiveAvgPool1d(20)  # Reduce to fixed 20 features
        self.fc1 = nn.Linear(64 * 20, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate model
model = MEG_CNN(n_channels=top_n_channels, n_times=X_cnn.shape[2])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# ===========================================
# 4. Train Model
# ===========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 30

train_losses = []
test_losses = []

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # Validation loss
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    test_loss = val_loss / len(test_loader)
    test_losses.append(test_loss)
    val_acc = correct / total

    print(f"Epoch {epoch+1}/{n_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {test_loss:.4f}, Val Acc: {val_acc:.3f}")

# ===========================================
# 5. Plot Loss Curves
# ===========================================
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('CNN Training vs Validation Loss')
plt.show()

# ===========================================
# 6. Final Test Accuracy
# ===========================================
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print(f"✅ Final Test Accuracy: {correct/total:.3f}")
